# 14장 고윳값·고유벡터와 PCA — 데이터의 주축 찾기 — 수리 도감 코랩 실습

원자료: 연구소 수학 노트 MATH-14 「Eigenvalues & Eigenvectors + PCA Mathematics」 · 해설 페이지: https://adminmrobo.github.io/math_basic_for_ai_dictionary/14_eigen_pca.html

위에서부터 셀을 차례로 실행하세요(Shift + Enter). 그래프는 영어 글자로 그렸습니다(코랩에 한글 글꼴을 따로 깔지 않아도 되게).

글 · 안상선 ((주)M-Robo 대표)


## 1. 고윳값: 근의 공식 vs numpy, 그리고 검산


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

A = np.array([[4, 1], [1, 3]], dtype=float)
lam = [(7 + np.sqrt(5)) / 2, (7 - np.sqrt(5)) / 2]
print("근의 공식 λ =", np.round(lam, 4))
vals, vecs = np.linalg.eig(A)
vals, vecs = np.real(vals), np.real(vecs)      # 버전에 따라 복소수 꼴로 나올 수 있어 실수부만 (대칭 행렬이라 실제로 실수)
print("np.linalg.eig λ =", np.round(vals, 4))
for i in range(2):
    v = vecs[:, i]
    print(f"λ={vals[i]:.4f}, v={v.round(4)}: A@v = {(A @ v).round(4)}, λv = {(vals[i] * v).round(4)}")
print("합", round(vals.sum(), 4), "= 대각합", np.trace(A), "/ 곱", round(vals.prod(), 4), "= 행렬식", round(np.linalg.det(A), 4))

## 2. 원을 타원으로: 고유벡터는 방향이 그대로


In [ ]:
t = np.linspace(0, 2 * np.pi, 100); circle = np.vstack([np.cos(t), np.sin(t)])
ellipse = A @ circle
plt.figure(figsize=(5.5, 5.5)); plt.plot(*circle, "gray", label="unit circle"); plt.plot(*ellipse, "b", label="A @ circle")
for i, c in enumerate(["red", "orange"]):
    v = vecs[:, i] * vals[i]; plt.arrow(0, 0, v[0], v[1], color=c, width=0.05, length_includes_head=True)
plt.gca().set_aspect("equal"); plt.legend(); plt.title("Eigenvectors keep their direction"); plt.grid(alpha=0.3); plt.show()

## 3. PCA 여섯 단계를 손으로


In [ ]:
X = np.array([[0, 0], [1, 1], [0, -1], [-1, 0]], dtype=float)
Xc = X - X.mean(axis=0)                                   # 1) 중심화 (여기서는 이미 평균 0)
S = Xc.T @ Xc / (len(X) - 1)                              # 2) 공분산 행렬
w, V = np.linalg.eigh(S)                                  # 3) 고윳값·고유벡터 (대칭 행렬용)
order = np.argsort(w)[::-1]; w, V = w[order], V[:, order] # 4) 내림차순
W = V[:, :1]                                              # 5) 상위 1개
Z = Xc @ W                                                # 6) 투영
print("공분산 행렬 =\n", S.round(4))
print("고윳값 =", w.round(4), "/ PC1 =", W.ravel().round(4), "/ 설명 비율 =", round(w[0] / w.sum(), 4))
print("투영 Z =", Z.ravel().round(4))
sk = PCA(n_components=1).fit(X)
print("sklearn 설명 비율 =", sk.explained_variance_ratio_.round(4), "/ 투영 =", sk.transform(X).ravel().round(4), "(부호는 반대일 수 있음)")

## 4. 비스듬한 점 구름과 주성분


In [ ]:
rng = np.random.default_rng(5)
P = rng.multivariate_normal([0, 0], [[3, 2], [2, 2]], size=200)
pca = PCA(2).fit(P)
plt.figure(figsize=(6, 6)); plt.scatter(P[:, 0], P[:, 1], s=10, alpha=0.5, label="data")
for comp, var, c in zip(pca.components_, pca.explained_variance_, ["red", "orange"]):
    plt.arrow(0, 0, *(comp * 2 * np.sqrt(var)), color=c, width=0.06, length_includes_head=True)
proj = (P @ pca.components_[0])[:, None] * pca.components_[0]
plt.scatter(proj[:, 0], proj[:, 1], s=8, color="green", label="projected on PC1")
plt.gca().set_aspect("equal"); plt.legend(); plt.title(f"PC1 explains {pca.explained_variance_ratio_[0]:.0%}"); plt.show()

## 5. 손글씨 숫자 64차원 -> 2차원


In [ ]:
digits = load_digits()
p2 = PCA(2).fit(digits.data); Z2 = p2.transform(digits.data)
print("손글씨 64차원: 상위 2개 설명 비율 =", round(p2.explained_variance_ratio_.sum(), 3),
      "/ 상위 10개 =", round(PCA(10).fit(digits.data).explained_variance_ratio_.sum(), 3))
plt.figure(figsize=(6.5, 5)); sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=digits.target, cmap="tab10", s=8)
plt.colorbar(sc, label="digit"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Digits (64-D) projected to 2-D"); plt.show()

## 생성 AI 실습 프롬프트

아래 글을 코랩의 Gemini 창(또는 다른 생성 AI)에 붙여 넣고, 만들어진 코드를 위 코드의 결과와 비교해 보세요.

```text
너는 고등학생에게 고윳값과 PCA를 가르치는 파이썬 선생님이야. 구글 코랩에서 실행되는 코드를 만들어 줘.
주제: 고윳값·고유벡터, 공분산 행렬, PCA
요구 사항:
1. A = [[4, 1], [1, 3]]의 고윳값을 (a) 특성방정식 λ² − 7λ + 11 = 0을 근의 공식으로 (b) np.linalg.eig로 구해 비교하고, 각 고유벡터 v에 대해 A @ v와 λ * v가 같은지 확인해 줘. 고윳값 합 = 대각합, 곱 = 행렬식도 확인해 줘.
2. 단위원 위 점 100개에 A를 곱해 타원이 되는 그림을 그리고, 두 고유벡터를 화살표로 표시해 줘(축 비율 같게, 영어 제목).
3. X = [[0,0],[1,1],[0,-1],[-1,0]]으로 PCA 여섯 단계(중심화 → 공분산 행렬(n-1) → 고윳값·고유벡터 → 내림차순 정렬 → 상위 1개 선택 → 투영)를 직접 구현하고, 설명된 분산 비율(75%)을 출력해 줘. sklearn PCA 결과와도 비교해 줘(부호가 다를 수 있음을 주석으로).
4. 비스듬한 2D 정규분포 점 200개(seed 고정)를 만들어 원래 점, PC1·PC2 화살표, PC1 위로 투영한 점을 한 그림에 그려 줘.
5. sklearn의 load_digits(8×8 손글씨 숫자, 특성 64개)를 PCA로 2차원으로 줄여 숫자별 색으로 산점도를 그리고, 상위 2개·10개 주성분의 설명된 분산 비율을 출력해 줘.
6. 모든 줄에 한국어 주석.
```
